# 10. Three-phase houses: phase configuration and phase pairing

**Goal.** Before testing Volt-VAr on three-phase houses (notebooks 11 and 12), establish from the **net meter only**:

1. **Configuration:** is the PV a single-phase inverter on one house phase, or a three-phase inverter?
2. **Phase:** for a single-phase inverter, which house phase is it on?

and check both against a reference built from the separately metered PV circuit(s).

**Cohorts describe the monitoring layout, not the inverter.** `3L1P` = three-phase house with **one PV measuring
circuit (CT)**; `3L3P` = three-phase house with three PV CTs. A three-phase inverter monitored with one CT is labelled
`3L1P`. So the inverter type is taken from the PV circuits' own output (section 4b), not from the label.

| Step | Data | Blind? |
|---|---|---|
| 4. Reference phase: the house phase whose voltage matches the PV circuit's voltage | `ami_raw_phaseseparate` (PV + house circuits) | No (answer key) |
| 4b. Reference configuration: PV circuit output ÷ rating, and balance across PV circuits | `ami_raw_phaseseparate` (PV circuits) + model as a hint | No (answer key) |
| 5. Blind configuration: how midday export splits across the three house phases | `ami_meter_residential` (net meter only) + rating | **Yes** |
| 6. Blind phase: the house phase with the export | `ami_meter_residential` | **Yes** |

**Rule for disagreements.** Sites are never excluded because the blind result disagrees with the reference (that would
hide the blind method's errors). Sites are excluded only for reference-quality reasons decided without looking at the
blind result (voltage match not confident, PV circuit covering part of the inverter, conflicting PV evidence).

Thresholds are chosen on **2024** and reported on **2025**. AS/NZS 4777.2:2020 Clause 2.11 requires balanced output
from three-phase inverters; Clause 3.3.2.1 makes them respond to the average of the three phase voltages.
Library: `lib/ami_phase_vvar.py` (`ami_fleet_vvar` is unchanged).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_phase_vvar as Phase

## 1. Settings

In [ ]:
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
TUNE_YEAR, REPORT_YEAR = 2024, 2025

NIGHT_HOURS = (0, 4)          # AEST, monthly night baseline
DAY_HOURS = (10, 14)          # AEST, midday window for the export signature and PV output

# 4. Reference phase pairing (voltage matching)
SAMPLE_EVERY_MIN = 15         # use every 15th minute for voltage matching
V_PLAUSIBLE = (150, 300)      # voltage readings outside this range are dropped
DV_RATIO_MIN = 2.0            # month confident if second-best mean |dV| >= 2x the best ...
MAX_BEST_DV = 1.0             # ... and the best mean |dV| <= 1 V
MIN_COMMON = 100              # common timestamps needed in a month
CONSENSUS_MIN_SHARE = 0.8     # site-level: share of confident months agreeing on the same phase
REF_DV_RATIO_MIN = 5.0        # notebook 11: site median dv_ratio needed to use the phase as a reference

# 4b. Reference configuration (PV circuits' own midday output, 90th percentile / rating)
PV_FULL_MIN = 0.55            # one PV circuit at or above this -> single-phase inverter, fully monitored
PV_PARTIAL_MAX = 0.33         # one PV circuit at or below this -> three-phase inverter, CT on one phase
PV_BALANCE_MIN_SHARE = 0.2    # three PV circuits, every share >= this -> three-phase inverter
PV_MIN_OVER_RATING = 0.05     # months with less PV output than this are ignored (no PV signal)

# 5. Blind configuration. Fixed values; the sweep on TUNE_YEAR is shown for reference.
# 0.6 is preferred over the sweep's best: overall correctness is flat across the grid, and 0.6 is more
# accurate on the months it decides (wrongly deciding is worse than leaving a month undecided).
AUTO_TUNE = False             # True: use the sweep's best setting instead of the values below
SHARE_1P = 0.6                # top phase share >= this -> single-phase inverter on that phase
MIN_SHARE_3P = 0.1            # all three shares >= this -> three-phase inverter
MIN_SIGNAL_FRAC = 0.1         # total midday export signal below 10% of rating -> no_signal
GRID_SHARE_1P = [0.5, 0.55, 0.6, 0.65, 0.7, 0.8]
GRID_MIN_SHARE_3P = [0.1, 0.15, 0.2, 0.25, 0.3]

# 7. Downstream selection (conservative): a site goes to notebooks 11/12 only if ALL evidence agrees
MIN_DECIDED_MONTHS = 2        # months with a blind decision (1P/3P) needed per site
MIN_VOTE_SHARE = 0.9          # share of decided months with the site's blind configuration
INCLUDE_PARTIAL = False       # three-phase inverters monitored on one phase (reference = one phase): keep out
REQUIRE_MODEL_CONFIRMATION = False   # True: also require a positive model hint (only ~40% of models give one)

DUCKDB_MEMORY, DUCKDB_THREADS = "8GB", 4
OVERWRITE = False

OUT_DIR = Fleet.store_dir() / "vvar_phase" / "phase_config"
OUT_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ["site_id", "year", "month"]
REF_TO_CONFIG = {"1P": "1P", "3P": "3P", "3P_partial": "3P"}   # reference config -> inverter type graded
print(f"Outputs -> {OUT_DIR}")

## 2. Cohorts (monitoring layout)

In [ ]:
# Below is a count of circuits
# 3L will be a house with 3 ac_load_net
# 1P will be a site with a 1 pv_circuit
## In this case, it MIGHT be a single-phase inverter monitored, OR
## a 3-phase inverter monitored in only 1 phase. If that's the case, I suspect that's 1/3 of the output

site_meta = pd.read_parquet(Fleet.Config.store_path("ami_site_metadata_residential"))
eligibility = Phase.build_phase_eligibility(site_meta)
display(pd.crosstab(eligibility.cohort.fillna("other"), eligibility.eligibility, margins=True))

sites = eligibility.loc[eligibility.eligibility == "eligible", ["site_id", "cohort", "rating_kw"]].reset_index(drop=True)
sites["label_config"] = sites.cohort.map({"3L1P": "1P", "3L3P": "3P"})     # what the label implies
print(f"{len(sites):,} eligible sites:", sites.cohort.value_counts().to_dict())

## 3. Interval queries (cached per month)

Three monthly queries over the eligible sites, cached in `phase_config/<name>/` (delete a file to re-query that month):
- `voltage_match_v2` (voltages; implausible readings dropped), 
- `signature` (net-meter midday export per house phase),
- `pv_output` (PV circuits' own midday output).

In [ ]:
def run_monthly(name: str, sql_fn, table: str) -> pd.DataFrame:
    folder = OUT_DIR / name
    folder.mkdir(exist_ok=True)
    parts = []
    for month in MONTHS:
        path = folder / f"{month}.parquet"
        if path.exists() and not OVERWRITE:
            parts.append(pd.read_parquet(path))
            continue
        if not Fleet.month_dir(table, month).exists():
            print(f"{month}: no {table} partition, skipped")
            continue
        t0 = time.time()
        con = Fleet.connect(DUCKDB_MEMORY, DUCKDB_THREADS)
        try:
            con.register("_sites", sites[["site_id", "rating_kw"]])
            df = con.execute(sql_fn(month)).df()
        finally:
            con.close()
        df["year"], df["month"] = int(month[:4]), int(month[5:])
        df.to_parquet(path, index=False)
        parts.append(df)
        print(f"{name} {month}: {len(df):,} rows ({time.time() - t0:.1f} s)")
    return pd.concat(parts, ignore_index=True)


# voltage match gets the delta from the reference phase, and the best match is the one with the lowest mean |dV|. 
# The second-best match is used to check if the best match is confident enough (the ratio of the two means must be >= DV_RATIO_MIN). 
# The best match must also have a mean |dV| <= MAX_BEST_DV. 
# A month is considered confident if it has at least MIN_COMMON common timestamps between the reference and the best match.
vm_all = run_monthly("voltage_match_v2", lambda m: Phase.voltage_match_sql(m, SAMPLE_EVERY_MIN, V_PLAUSIBLE),
                     "ami_raw_phaseseparate")

# how midday export from the net meter splits across the three house phases. 
# If almost all of it is on one phase, that’s a single-phase inverter on that phase
# if it’s spread over all three, a three-phase inverter.
sig_all = run_monthly("signature", lambda m: Phase.phase_signature_sql(m, night_hours=NIGHT_HOURS, day_hours=DAY_HOURS),
                      "ami_meter_residential")

# how much the PV circuits produce at midday, compared with the rating
pv_all = run_monthly("pv_output", lambda m: Phase.pv_output_sql(m, day_hours=DAY_HOURS), "ami_raw_phaseseparate")

print(f"voltage match: {len(vm_all):,} rows; signature: {len(sig_all):,}; pv output: {len(pv_all):,}")

In [ ]:
# One row per site x month x PV circuit x house circuit
print(f"{len(vm_all):,} rows")
display(vm_all.head(10))

## 4. Reference phase pairing (voltage matching)

Assuming the PV circuit and the house phase it is wired to measure nearly the same voltage. 

Per site-month and PV circuit: the best-matching house phase, its mean |ΔV| and the ratio to the second best. 

**Confident** = ratio ≥ `DV_RATIO_MIN` and best ≤ `MAX_BEST_DV`. 

Site level: the phase chosen in most confident months, kept if at least `CONSENSUS_MIN_SHARE` of them agree. 

For three PV circuits, the pairing must also be one-to-one (three different house phases).

In [ ]:
truth_month = Phase.assign_phase_truth(vm_all, ratio_min=DV_RATIO_MIN, max_best_dv=MAX_BEST_DV,
                                       min_common=MIN_COMMON).merge(sites[["site_id", "cohort"]], on="site_id")
truth_site = Phase.consensus_truth(truth_month).merge(sites[["site_id", "cohort"]], on="site_id")
truth_site["ref_ok"] = truth_site.agree_share >= CONSENSUS_MIN_SHARE

# Median dv_ratio over confident months (strength of the voltage reference, used for notebook 11)
dv_site = (truth_month[truth_month.confident].groupby(["site_id", "pv_circuit_id"]).dv_ratio.median()
           .rename("dv_ratio_median").reset_index())
truth_site = truth_site.merge(dv_site, on=["site_id", "pv_circuit_id"], how="left")

o2o = (truth_site[truth_site.ref_ok].groupby("site_id")
       .agg(n_pv=("pv_circuit_id", "nunique"), n_load=("load_circuit_id", "nunique")).reset_index())
o2o["one_to_one"] = (o2o.n_pv == 3) & (o2o.n_load == 3)
truth_site = truth_site.merge(o2o[["site_id", "one_to_one"]], on="site_id", how="left")
truth_site["one_to_one"] = truth_site.one_to_one.fillna(False).astype(bool)

print("PV-circuit-months by confidence:")
display(truth_month.groupby("cohort").agg(pv_circuit_months=("confident", "size"), confident=("confident", "mean"),
                                          median_best_dv=("best_dv", "median"), median_ratio=("dv_ratio", "median"))
        .round(3))
display(pd.DataFrame({
    "eligible sites": sites.groupby("cohort").size(),
    "sites with a reference phase": truth_site[truth_site.ref_ok].groupby("cohort").site_id.nunique(),
    "3 PV circuits, one-to-one": truth_site[truth_site.one_to_one].groupby("cohort").site_id.nunique(),
}).fillna(0).astype(int))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), dpi=120)
for cohort, c in [("3L1P", "#D85A30"), ("3L3P", "#1f4e79")]:
    t = truth_month[truth_month.cohort == cohort]
    axes[0].hist(t.best_dv.clip(upper=3), bins=60, histtype="step", color=c, label=f"{cohort} best")
    axes[0].hist(t.second_dv.clip(upper=3), bins=60, histtype="step", color=c, ls="--", label=f"{cohort} second")
    axes[1].hist(np.log10(t.dv_ratio.clip(lower=0.5, upper=1000)), bins=60, histtype="step", color=c, label=cohort)
axes[0].set(xlabel="Mean |V_pv − V_load| (V, clipped at 3)", ylabel="PV-circuit-months", title="Best and second-best match")
axes[1].axvline(np.log10(DV_RATIO_MIN), color="k", lw=0.8, ls=":")
axes[1].set(xlabel="log10(second / best)", title=f"Separation (dotted: ratio = {DV_RATIO_MIN:g})")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Example site-months for the voltage-matching reference, by category, for the daily plots.
N_PER_CATEGORY = 5          # examples per category
EX_SEED = 42

tm = truth_month.merge(truth_site[["site_id", "pv_circuit_id", "agree_share", "ref_ok", "one_to_one"]],
                       on=["site_id", "pv_circuit_id"], how="left")

categories = {
    # passed: very clear match (ratio >= 20, best within 0.1 V)
    "pass_clear":            tm.confident & (tm.dv_ratio >= 20) & (tm.best_dv <= 0.1),
    # passed, but only just (ratio between 2 and 3)
    "pass_borderline":       tm.confident & (tm.dv_ratio < 3),
    # failed: two house phases about equally close (the bump at ratio ~1)
    "fail_ambiguous":        ~tm.confident & (tm.dv_ratio < 1.2),
    # failed: even the best match is more than 1 V away
    "fail_far":              ~tm.confident & (tm.best_dv > MAX_BEST_DV),
    # confident this month, but the site's months disagree (no reliable site-level phase)
    "site_inconsistent":     tm.confident & (tm.ref_ok == False),
    # three-phase site whose PV circuits do not map to three different house phases
    "3l3p_not_one_to_one":   (tm.cohort == "3L3P") & tm.confident & ~tm.one_to_one.fillna(False).astype(bool),
}

parts = []
for name, mask in categories.items():
    pool = tm[mask].drop_duplicates(["site_id", "year", "month"])
    pick = pool.sample(n=min(N_PER_CATEGORY, len(pool)), random_state=EX_SEED)
    parts.append(pick.assign(category=name, n_in_category=len(pool)))
examples = (pd.concat(parts, ignore_index=True)
            [["category", "n_in_category", "cohort", "site_id", "year", "month", "pv_circuit_id",
              "load_circuit_id", "best_dv", "second_dv", "dv_ratio", "confident", "agree_share"]]
            .round(3))

examples.to_csv(OUT_DIR / "voltage_match_examples.csv", index=False)
display(examples)
print(f"Saved {len(examples)} examples -> {OUT_DIR / 'voltage_match_examples.csv'}")

In [ ]:
from bms_sa_review.ami_analysis.lib import ami_daily_plots as Daily

PLOT_CATEGORY = "pass_clear"     # change to any category above

for _, r in examples[examples.category == PLOT_CATEGORY].iterrows():
    month = f"{int(r.year)}-{int(r.month):02d}"
    print(f"--- {r.category} | {r.cohort} site {r.site_id} {month} | best {r.best_dv} V, "
          f"second {r.second_dv} V, ratio {r.dv_ratio}")
    Daily.show_site_day(int(r.site_id), month=month, day_hours=DAY_HOURS,
                        blind_month=blind_month, truth_month=truth_month)

In [ ]:
import duckdb
import pandas as pd
from bms_sa_review.synthetic_ami_creation.config import ami_config as Config

SITE = 2145669016
STORE = Config.store_path("ami_meter_residential").parent

meta = pd.read_parquet(Config.store_path("ami_circuit_metadata"))
circ = meta[meta.site_id == SITE][["circuit_id", "circuit_type", "device_type", "min_time", "max_time"]]
display(circ)                                   # what the metadata says exists
ids = ", ".join(map(str, circ.circuit_id))

con = duckdb.connect()
local = []
for table, glob in [("ami_extract", "*.parquet"), ("ami_raw_phaseseparate", "*.parquet"),
                    ("ami_meter_residential", "*.parquet")]:
    local.append(con.execute(f"""
        SELECT '{table}' AS tbl, dt_month, circuit_id, count(*) AS n_rows
        FROM read_parquet('{(STORE / table).as_posix()}/dt_month=*/{glob}', hive_partitioning = 1, union_by_name = true)
        WHERE circuit_id IN ({ids})
        GROUP BY ALL
    """).df())
local = pd.concat(local, ignore_index=True)
display(local.pivot_table(index="dt_month", columns=["tbl", "circuit_id"], values="n_rows", fill_value=0))

In [ ]:
from bms_sa_review.synthetic_ami_creation.lib import ami_athena as Athena

YEAR = 2025
sql = f"""
    SELECT month, circuit_id, is_pv, count(*) AS n_rows, min(t_stamp) AS first_ts, max(t_stamp) AS last_ts
    FROM ts
    WHERE year = {YEAR} AND circuit_id IN ({ids})
    GROUP BY 1, 2, 3
    ORDER BY 2, 1
"""
Athena.preview_sql(sql)
src = Athena.aq(sql, database=Config.SAI, label=f"site {SITE} circuits {YEAR}")
display(src.pivot_table(index="month", columns="circuit_id", values="n_rows", fill_value=0))
Athena.scan_report()

## 4b. Reference configuration (what the PV circuits say)

Per site, the median over months of the PV circuits' own midday output (90th percentile, summed over PV circuits) ÷
rating, and for three PV circuits the smallest circuit's share:

| PV circuits | Rule | Reference configuration |
|---|---|---|
| 1 | output ÷ rating ≥ `PV_FULL_MIN` | `1P` single-phase inverter, fully monitored |
| 1 | output ÷ rating ≤ `PV_PARTIAL_MAX` **and** the model says three-phase | `3P_partial` three-phase inverter, CT on one phase |
| 1 | in between | `uncertain` |
| 3 | every share ≥ `PV_BALANCE_MIN_SHARE` | `3P` three-phase inverter, fully monitored |
| 3 | otherwise | `multi` unbalanced (e.g. several inverters) |

The inverter model gives a hint (`Phase.MODEL_HINTS`, e.g. Symo → 3P, Primo/"1P" → 1P). It never decides the
configuration on its own, but when it **contradicts** the PV-circuit rule the site is set to `uncertain`. Low PV output
alone cannot separate "one phase of a three-phase inverter" from "a single-phase inverter with low output" (both give
about 0.2-0.45 of the rating), so `3P_partial` additionally **requires** a three-phase model; without one the site is
`uncertain`. Sites with no usable PV output
get `no_pv_data`. Only `1P`, `3P` and `3P_partial` are used
to grade the blind method; `uncertain`, `multi` and `no_pv_data` are reported and left out.

In [ ]:
ref = Phase.reference_configuration(pv_all, sites, full_min=PV_FULL_MIN, partial_max=PV_PARTIAL_MAX,
                                    balance_min_share=PV_BALANCE_MIN_SHARE, min_pv_over_rating=PV_MIN_OVER_RATING)
site_ref = (sites.merge(ref, on="site_id", how="left")
            .merge(site_meta[["site_id", "manufacturer", "model", "inverter_count"]], on="site_id", how="left"))
site_ref["ref_config"] = site_ref.ref_config.fillna("no_pv_data")
site_ref["model_hint"] = site_ref.model.map(Phase.model_hint)

# Conflicting non-blind evidence -> uncertain (e.g. a low-output single-phase inverter would otherwise look
# like one phase of a three-phase inverter). Uses only the PV circuits and the model, never the net meter.
conflict = (((site_ref.ref_config == "3P_partial") & (site_ref.model_hint != "3P"))
            | ((site_ref.ref_config == "1P") & (site_ref.model_hint == "3P"))
            | ((site_ref.ref_config == "3P") & (site_ref.model_hint == "1P")))
site_ref["ref_rule"] = site_ref.ref_config
site_ref.loc[conflict, "ref_config"] = "uncertain"
print(f"{conflict.sum()} sites set to 'uncertain': the model contradicts the PV-circuit rule, or a "
      "'3P_partial' site has no three-phase model to confirm it:")
display(site_ref[conflict][["site_id", "cohort", "ref_rule", "model", "model_hint", "pv_over_rating", "min_pv_share"]].round(2))
site_ref["ref_true"] = site_ref.ref_config.map(REF_TO_CONFIG)            # graded inverter type (NaN = not graded)

print("Monitoring layout (label) vs reference configuration (sites):")
display(pd.crosstab(site_ref.cohort, site_ref.ref_config, margins=True))
print("Reference configuration vs model hint (sites; '-' = no hint from the model string):")
display(pd.crosstab(site_ref.ref_config, site_ref.model_hint.fillna("-"), margins=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), dpi=120)
one = site_ref[site_ref.n_pv_circuits == 1]
axes[0].hist(one.pv_over_rating.clip(upper=1.5), bins=60, color="#D85A30", alpha=0.8)
axes[0].axvline(PV_PARTIAL_MAX, color="k", ls=":", lw=0.9, label=f"PV_PARTIAL_MAX = {PV_PARTIAL_MAX:g}")
axes[0].axvline(PV_FULL_MIN, color="k", ls="--", lw=0.9, label=f"PV_FULL_MIN = {PV_FULL_MIN:g}")
axes[0].set(xlabel="PV circuit midday output ÷ rating (site median, clipped at 1.5)", ylabel="Sites",
            title="One PV circuit: full inverter or one phase of it?")
three = site_ref[site_ref.n_pv_circuits == 3]
axes[1].hist(three.min_pv_share, bins=50, color="#1f4e79", alpha=0.8)
axes[1].axvline(PV_BALANCE_MIN_SHARE, color="k", ls=":", lw=0.9, label=f"PV_BALANCE_MIN_SHARE = {PV_BALANCE_MIN_SHARE:g}")
axes[1].set(xlabel="Smallest PV circuit's share of total PV output (site median)", title="Three PV circuits: balanced?")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "reference_configuration.png", bbox_inches="tight")
plt.show()

## 5. Blind configuration (net meter only)

Per site-month, from the midday 90th percentile of `P_est = P_base − P_net` on each house phase: each phase's share of
the export signal, then

| Rule | Configuration |
|---|---|
| not exactly three house circuits that month | `incomplete` |
| total signal < `MIN_SIGNAL_FRAC` × rating | `no_signal` |
| top share ≥ `SHARE_1P` | `1P` (on the top phase) |
| every share ≥ `MIN_SHARE_3P` | `3P` |
| otherwise | `unclear` |

Graded against the **reference configuration** (4b). The label-based confusion is shown for comparison.

In [ ]:
def classify(sig: pd.DataFrame, share_1p: float, min_share_3p: float) -> pd.DataFrame:
    c = Phase.classify_configuration(sig, sites, share_1p=share_1p, min_share_3p=min_share_3p,
                                     min_signal_frac=MIN_SIGNAL_FRAC)
    return c.merge(site_ref[["site_id", "cohort", "label_config", "ref_config", "ref_true"]], on="site_id")


# Tune on TUNE_YEAR, graded against the reference configuration (undecided counts as not correct)
sig_tune = sig_all[sig_all.year == TUNE_YEAR]
rows = []
for a in GRID_SHARE_1P:
    for b in GRID_MIN_SHARE_3P:
        c = classify(sig_tune, a, b)
        c = c[c.ref_true.notna()]
        decided = c.blind_config.isin(["1P", "3P"])     # incomplete / no_signal / unclear = undecided
        correct = c.blind_config == c.ref_true
        rows.append({"share_1p": a, "min_share_3p": b, "decided": decided.mean(),
                     "accuracy_decided": correct[decided].mean(), "correct_overall": correct.mean(),
                     "correct_ref1P": correct[c.ref_true == "1P"].mean(), "correct_ref3P": correct[c.ref_true == "3P"].mean()})
sweep = pd.DataFrame(rows)
sweep["dist_to_default"] = (sweep.share_1p - SHARE_1P).abs() + (sweep.min_share_3p - MIN_SHARE_3P).abs()
sweep = sweep.sort_values(["correct_overall", "dist_to_default"], ascending=[False, True])
display(sweep.head(10).round(3))

best = sweep.iloc[0]
print(f"Sweep best on {TUNE_YEAR}: share_1p = {best.share_1p}, min_share_3p = {best.min_share_3p}")
if AUTO_TUNE:
    SHARE_1P, MIN_SHARE_3P = float(best.share_1p), float(best.min_share_3p)
print(f"Used: SHARE_1P = {SHARE_1P}, MIN_SHARE_3P = {MIN_SHARE_3P}")
display(sweep[(sweep.share_1p == SHARE_1P) & (sweep.min_share_3p == MIN_SHARE_3P)].round(3))

In [ ]:
blind_month = classify(sig_all, SHARE_1P, MIN_SHARE_3P)
graded = blind_month[blind_month.ref_true.notna()]

for year in (TUNE_YEAR, REPORT_YEAR):
    b = graded[graded.year == year]
    tag = " (tuning year)" if year == TUNE_YEAR else " (reported)"
    print(f"\n--- {year}{tag}: site-months, by reference configuration ---")
    display(pd.crosstab(b.ref_config, b.blind_config, margins=True))
    display(pd.crosstab(b.ref_config, b.blind_config, normalize="index").round(3))
    dec = b[b.blind_config.isin(["1P", "3P"])]
    print(f"Accuracy on decided months: {(dec.blind_config == dec.ref_true).mean():.1%} of {len(dec):,}")

print(f"\n--- For comparison, {REPORT_YEAR} graded against the monitoring label (n_pv_phases) ---")
b = blind_month[blind_month.year == REPORT_YEAR]
display(pd.crosstab(b.cohort, b.blind_config, normalize="index").round(3))

# Site level: majority vote over decided months
blind_site = (Phase.site_vote(blind_month)
              .merge(site_ref[["site_id", "cohort", "ref_config", "ref_true"]], on="site_id", how="right"))
blind_site["blind_config"] = blind_site.blind_config.fillna("undecided")
print("\n--- Site level (all months), by reference configuration ---")
display(pd.crosstab(blind_site.ref_config, blind_site.blind_config, margins=True))

In [ ]:
# Interesting site-months in the reported year: wrong calls, plus clear correct ones for contrast.
# Errors are sorted by how confident the wrong call was (furthest from the threshold first).
N_PER_CASE = 5

g = graded[(graded.year == REPORT_YEAR) & graded.blind_config.isin(["1P", "3P"])].copy()
g["case"] = np.select([g.blind_config == g.ref_true, g.blind_config == "1P"],
                      ["correct_" + g.blind_config, "blind1P_ref3P"], default="blind3P_ref1P")
g["month_str"] = g.year.astype(str) + "-" + g.month.astype(int).map("{:02d}".format)

order = {"blind1P_ref3P": ("top_share", False),   # said 1P with high confidence, reference says 3P
         "blind3P_ref1P": ("min_share", False),   # said 3P with high confidence, reference says 1P
         "correct_1P":    ("top_share", False),
         "correct_3P":    ("min_share", False)}
cases = pd.concat([g[g.case == k].sort_values(col, ascending=asc).drop_duplicates("site_id").head(N_PER_CASE)
                   for k, (col, asc) in order.items()])

print(g.case.value_counts())
display(cases[["case", "site_id", "month_str", "cohort", "ref_config", "blind_config",
               "top_share", "min_share", "top_over_rating", "total", "rating_kw"]].round(3))
cases.to_csv("nb10_review_cases.csv", index=False)   # optional

In [ ]:
from bms_sa_review.ami_analysis.lib import ami_daily_plots as Daily

r = cases[cases.case == "blind3P_ref1P"].iloc[0]

# What the blind classifier sees: P_est per house phase, monthly signal, shares vs thresholds
Daily.show_export_signature(int(r.site_id), r.month_str, share_1p=SHARE_1P,
                            min_share_3p=MIN_SHARE_3P, min_signal_frac=MIN_SIGNAL_FRAC)

# What the reference sees: raw circuits and PV-vs-house voltage matching on the best PV day
Daily.show_site_day(int(r.site_id), month=r.month_str, blind_month=blind_month, truth_month=truth_month)

## Daily plots

In [ ]:
from bms_sa_review.ami_analysis.lib import ami_daily_plots as Daily

SITE_ID, MONTH, DATE = 674258435, "2025-01", None
df_day = Daily.show_site_day(SITE_ID, date=DATE, month=MONTH, day_hours=DAY_HOURS,
                             blind_month=blind_month, truth_month=truth_month)
est_month = Daily.show_export_signature(SITE_ID, MONTH, DATE, night_hours=NIGHT_HOURS, day_hours=DAY_HOURS,
                                        share_1p=SHARE_1P, min_share_3p=MIN_SHARE_3P,
                                        min_signal_frac=MIN_SIGNAL_FRAC)

## 6. Phase identification (reference `1P` sites only)

For single-phase inverters (reference `1P`) with a confident voltage pairing: 
- does the blind phase (top export share) equal the voltage-matched phase? 

Three-phase inverters monitored on one phase are left out.

In [ ]:
ref1p = site_ref[site_ref.ref_config == "1P"].site_id
ref_month = truth_month[truth_month.confident & truth_month.site_id.isin(ref1p)][KEYS + ["load_circuit_id"]]
pm = blind_month[blind_month.site_id.isin(ref1p) & (blind_month.blind_config == "1P")].merge(ref_month, on=KEYS)
pm["phase_correct"] = pm.blind_load_circuit_id == pm.load_circuit_id
display(pm.groupby("year").agg(site_months=("phase_correct", "size"), phase_correct=("phase_correct", "mean")).round(3))

ref_site = truth_site[truth_site.site_id.isin(ref1p) & truth_site.ref_ok][["site_id", "load_circuit_id"]]
ps = blind_site[blind_site.site_id.isin(ref1p) & (blind_site.blind_config == "1P")].merge(ref_site, on="site_id")
ps["phase_correct"] = ps.blind_load_circuit_id == ps.load_circuit_id
print(f"Site level: {ps.phase_correct.mean():.1%} of {len(ps):,} reference-1P sites get the right phase")
wrong = pm[~pm.phase_correct]
print(f"{len(wrong):,} wrong site-months; median top share {wrong.top_share.median():.2f} "
      f"(correct: {pm[pm.phase_correct].top_share.median():.2f})")

In [ ]:
wrong = pm[~pm.phase_correct].sort_values("top_share", ascending=False).drop_duplicates("site_id")
wrong["month_str"] = wrong.year.astype(str) + "-" + wrong.month.astype(int).map("{:02d}".format)
display(wrong[["site_id", "month_str", "blind_load_circuit_id", "load_circuit_id", "top_share", "top_over_rating"]].head(10))

r = wrong.iloc[0]
Daily.show_export_signature(int(r.site_id), r.month_str, share_1p=SHARE_1P,
                            min_share_3p=MIN_SHARE_3P, min_signal_frac=MIN_SIGNAL_FRAC)
Daily.show_site_day(int(r.site_id), month=r.month_str, blind_month=blind_month, truth_month=truth_month)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), dpi=120)
styles = {"1P": "#D85A30", "3P": "#1f4e79", "3P_partial": "#1baf7a"}
for rc, c in styles.items():
    b = blind_month[(blind_month.ref_config == rc) & ~blind_month.blind_config.isin(["no_signal", "incomplete"])]
    axes[0].hist(b.top_share, bins=50, histtype="step", color=c, label=f"ref {rc}")
    axes[1].hist(b.min_share, bins=50, histtype="step", color=c, label=f"ref {rc}")
    axes[2].hist(b.top_over_rating.clip(upper=1.5), bins=60, histtype="step", color=c, label=f"ref {rc}")
axes[0].axvline(SHARE_1P, color="k", ls=":", lw=0.8)
axes[1].axvline(MIN_SHARE_3P, color="k", ls=":", lw=0.8)
axes[0].set(xlabel="Top-phase share of midday export", ylabel="Site-months")
axes[1].set(xlabel="Smallest phase share")
axes[2].set(xlabel="Top-phase export ÷ inverter rating (clipped at 1.5)")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "phase_signature_distributions.png", bbox_inches="tight")
plt.show()

## 7. Downstream selection (conservative)

Sections 5 and 6 grade the blind method on **all** graded sites (so its error rate is honest). 

For the Volt-VAr tests in notebooks 11 and 12, a site is used only if **every** piece of evidence points to the same configuration. 

Checks, in order. The first that fails is the site's `exclusion_reason`:

| # | Check | Excluded as |
|---|---|---|
| 1 | PV-circuit rule gives a clear answer | `ref_uncertain` / `ref_multi` / `ref_no_pv_data` |
| 2 | one-phase-of-three-phase only with a three-phase model; and only if `INCLUDE_PARTIAL` | `partial_no_model_confirmation` / `partial_excluded_by_setting` |
| 3 | monitoring label agrees with the PV-circuit rule (1 PV circuit → 1P, 3 → 3P) | `label_vs_ref` |
| 4 | model does not contradict it (if it gives a hint); with `REQUIRE_MODEL_CONFIRMATION`, it must confirm it | `model_conflict` / `no_model_confirmation` |
| 5 | at least `MIN_DECIDED_MONTHS` months with a blind decision | `too_few_decided_months` |
| 6 | blind site configuration agrees with the reference | `blind_disagrees` |
| 7 | blind configuration consistent over months (vote share ≥ `MIN_VOTE_SHARE`) | `blind_inconsistent` |
| 8 | single-phase: strong voltage reference (consensus, median `dv_ratio` ≥ `REF_DV_RATIO_MIN`) | `weak_phase_reference` |
| 9 | single-phase: blind phase = voltage-matched phase | `phase_disagrees` |

On net-metr data only sites with a confident blind configuration would be analysed, so the Volt-VAr results from 11/12
apply to that population; the configuration error itself is reported here (sections 5-6).

**Month level** (`month_ok`, for 11/12): a clean site's month is used only if that month's blind configuration equals
the site's, and for single-phase sites the blind phase is the reference phase with a confident voltage match that month.

In [ ]:
def _true(v) -> bool:
    return bool(v) if pd.notna(v) else False


dec_m = blind_month[blind_month.blind_config.isin(["1P", "3P"])]
votes = dec_m.groupby(["site_id", "blind_config"]).size().unstack(fill_value=0)

ev = site_ref[["site_id", "cohort", "rating_kw", "label_config", "ref_rule", "ref_config", "model", "model_hint",
               "pv_over_rating", "min_pv_share", "n_months_pv"]].copy()
ev["expected"] = ev.ref_rule.map(REF_TO_CONFIG)                    # configuration the PV evidence points to
ev["n_decided"] = ev.site_id.map(votes.sum(axis=1)).fillna(0).astype(int)
ev["blind_site"] = ev.site_id.map(votes.idxmax(axis=1))
ev["vote_share"] = ev.site_id.map(votes.max(axis=1) / votes.sum(axis=1))

ph = (truth_site.groupby("site_id")
      .agg(n_pv_paired=("pv_circuit_id", "nunique"), ref_ok_all=("ref_ok", "all"),
           dv_ratio_min=("dv_ratio_median", "min"), ref_phase=("load_circuit_id", "first"),
           one_to_one=("one_to_one", "first"))
      .reset_index())
ev = ev.merge(ph, on="site_id", how="left")
blind_phase = (dec_m[dec_m.blind_config == "1P"].groupby("site_id").blind_load_circuit_id
               .agg(lambda x: x.mode().iloc[0]))
ev["blind_phase"] = ev.site_id.map(blind_phase)


def exclusion_reason(x) -> str:
    if x.ref_rule in ("uncertain", "multi", "no_pv_data"):
        return f"ref_{x.ref_rule}"
    if x.ref_rule == "3P_partial":
        if x.model_hint != "3P":
            return "partial_no_model_confirmation"
        if not INCLUDE_PARTIAL:
            return "partial_excluded_by_setting"
    elif x.expected != x.label_config:
        return "label_vs_ref"
    if pd.notna(x.model_hint) and x.model_hint != x.expected:
        return "model_conflict"
    if REQUIRE_MODEL_CONFIRMATION and x.model_hint != x.expected:
        return "no_model_confirmation"
    if x.n_decided < MIN_DECIDED_MONTHS:
        return "too_few_decided_months"
    if x.blind_site != x.expected:
        return "blind_disagrees"
    if x.vote_share < MIN_VOTE_SHARE:
        return "blind_inconsistent"
    if x.expected == "1P":
        if not (_true(x.ref_ok_all) and pd.notna(x.dv_ratio_min) and x.dv_ratio_min >= REF_DV_RATIO_MIN):
            return "weak_phase_reference"
        if x.blind_phase != x.ref_phase:
            return "phase_disagrees"
    return "clean"


ev["exclusion_reason"] = ev.apply(exclusion_reason, axis=1)
clean = ev.exclusion_reason == "clean"
ev["use_in_11"] = clean & (ev.ref_rule == "1P")
ev["use_in_12"] = clean & (ev.ref_rule == "3P")
ev["use_in_12_partial"] = clean & (ev.ref_rule == "3P_partial")

print(f"Settings: MIN_DECIDED_MONTHS={MIN_DECIDED_MONTHS}, MIN_VOTE_SHARE={MIN_VOTE_SHARE}, "
      f"INCLUDE_PARTIAL={INCLUDE_PARTIAL}, REQUIRE_MODEL_CONFIRMATION={REQUIRE_MODEL_CONFIRMATION}")
display(pd.crosstab(ev.exclusion_reason, ev.ref_rule, margins=True))
print("Sites passed downstream:", ev[["use_in_11", "use_in_12", "use_in_12_partial"]].sum().to_dict())

# Sensitivity: clean sites by MIN_DECIDED_MONTHS (other checks unchanged)
def n_clean(k):
    keep = MIN_DECIDED_MONTHS
    globals()["MIN_DECIDED_MONTHS"] = k
    r = ev.apply(exclusion_reason, axis=1)
    globals()["MIN_DECIDED_MONTHS"] = keep
    return {"1P (-> 11)": int(((r == "clean") & (ev.ref_rule == "1P")).sum()),
            "3P (-> 12)": int(((r == "clean") & (ev.ref_rule == "3P")).sum())}
display(pd.DataFrame({f">= {k} months": n_clean(k) for k in (1, 2, 3, 6)}))

In [ ]:
# Month-level filter for notebooks 11/12
conf_1p = (truth_month[truth_month.confident & truth_month.site_id.isin(ev.loc[ev.expected == "1P", "site_id"])]
           [KEYS + ["load_circuit_id"]].drop_duplicates(KEYS)
           .rename(columns={"load_circuit_id": "ref_phase_month"}))
blind_month = (blind_month.drop(columns=["expected", "exclusion_reason", "ref_phase", "ref_phase_month", "month_ok"],
                                errors="ignore")
               .merge(ev[["site_id", "expected", "exclusion_reason", "ref_phase"]], on="site_id", how="left")
               .merge(conf_1p, on=KEYS, how="left"))
phase_ok = (blind_month.blind_load_circuit_id == blind_month.ref_phase) & (blind_month.ref_phase_month == blind_month.ref_phase)
blind_month["month_ok"] = ((blind_month.exclusion_reason == "clean") & (blind_month.blind_config == blind_month.expected)
                           & ((blind_month.expected == "3P") | phase_ok))

m = blind_month[blind_month.exclusion_reason == "clean"]
print("Site-months of clean sites, used vs dropped by the month-level filter:")
display(pd.crosstab([m.expected, m.year], m.month_ok, margins=True))

## 7b. Saved outputs

| File | Content |
|---|---|
| `site_evidence.parquet` | one row per site: all evidence, `exclusion_reason`, `use_in_11` / `use_in_12` / `use_in_12_partial` |
| `phase_pairs_site.parquet` | one row per (site, PV circuit): reference phase and pairing quality, plus the site flags |
| `blind_config_site_month.parquet` | blind result per site-month, with `month_ok` |
| `truth_pairing_site_month.parquet`, `reference_configuration_site.parquet` | reference details |

Notebooks 11 and 12 should read the flags from `site_evidence.parquet` and keep only `month_ok` months.
Shared result format for 11-13: `Phase.RESULT_COLUMNS`.

In [ ]:
pairs = (truth_site[["site_id", "pv_circuit_id", "load_circuit_id", "n_months_confident", "agree_share",
                     "dv_ratio_median", "ref_ok", "one_to_one"]]
         .rename(columns={"load_circuit_id": "ref_load_circuit_id"})
         .merge(ev[["site_id", "cohort", "rating_kw", "ref_rule", "ref_config", "expected", "blind_site", "vote_share",
                    "blind_phase", "exclusion_reason", "use_in_11", "use_in_12", "use_in_12_partial"]],
                on="site_id", how="right"))
pairs["ref_ok"] = pairs.ref_ok.fillna(False).astype(bool)
pairs["one_to_one"] = pairs.one_to_one.fillna(False).astype(bool)

ev.to_parquet(OUT_DIR / "site_evidence.parquet", index=False)
pairs.to_parquet(OUT_DIR / "phase_pairs_site.parquet", index=False)
blind_month.to_parquet(OUT_DIR / "blind_config_site_month.parquet", index=False)
truth_month.to_parquet(OUT_DIR / "truth_pairing_site_month.parquet", index=False)
site_ref.to_parquet(OUT_DIR / "reference_configuration_site.parquet", index=False)
print("Sites passed downstream:", ev[["use_in_11", "use_in_12", "use_in_12_partial"]].sum().to_dict())
print(f"Saved to {OUT_DIR}")

## 8. Checks

**8.1 Wrong phase: blind method or reference?** In reference-1P site-months where the blind phase differs from the
voltage-matched phase, is the voltage match weaker? If so, the reference is the likelier culprit.

**8.2 Ambiguous voltage matches by monitoring device.** PV-circuit-months whose best and second-best matches are almost
equal (`dv_ratio` < `AMBIGUOUS_RATIO`) suggest a device that applies one voltage reading to several channels.

**8.3 Where blind and reference disagree.** Site-level disagreements with the PV-circuit evidence and the model, to
inspect with the daily plots (`ami_daily_plots`). These stay in the results as blind-method errors.

In [ ]:
# 8.1 Wrong vs right phase months: strength of the voltage match
tm = truth_month[["site_id", "year", "month", "best_dv", "dv_ratio"]]
chk = pm.merge(tm, on=KEYS, how="left")
display(chk.groupby("phase_correct")[["best_dv", "dv_ratio", "top_share"]]
        .describe(percentiles=[0.1, 0.5, 0.9]).T.round(3))
for r in (5, 10, 20):
    strong = chk.dv_ratio >= r
    print(f"dv_ratio >= {r:>2}: phase correct in {chk[strong].phase_correct.mean():.1%} of {strong.sum():,} site-months "
          f"(below: {chk[~strong].phase_correct.mean():.1%} of {(~strong).sum():,})")

In [ ]:
# 8.2 Ambiguous voltage matches by monitoring device type of the PV circuit
AMBIGUOUS_RATIO = 1.2
circ = pd.read_parquet(Fleet.Config.store_path("ami_circuit_metadata"))[["circuit_id", "device_type"]]
dv = truth_month.merge(circ.rename(columns={"circuit_id": "pv_circuit_id"}), on="pv_circuit_id", how="left")
dv["ambiguous"] = dv.dv_ratio < AMBIGUOUS_RATIO
display(dv.groupby(["cohort", "device_type"]).agg(pv_circuit_months=("ambiguous", "size"),
                                                  ambiguous=("ambiguous", "mean"), confident=("confident", "mean"),
                                                  median_ratio=("dv_ratio", "median")).round(3))

In [ ]:
# 8.3 Site-level disagreements between the blind configuration and the reference configuration
dis = blind_site[blind_site.ref_true.notna() & blind_site.blind_config.isin(["1P", "3P"])
                 & (blind_site.blind_config != blind_site.ref_true)]
med = (blind_month.groupby("site_id")[["top_share", "min_share", "top_over_rating"]].median()
       .add_prefix("net_").reset_index())
dis_view = (dis[["site_id", "ref_config", "blind_config", "vote_share"]]
            .merge(site_ref[["site_id", "cohort", "rating_kw", "pv_over_rating", "min_pv_share", "model", "model_hint",
                             "n_months_pv"]], on="site_id")
            .merge(med, on="site_id", how="left")
            .sort_values(["ref_config", "pv_over_rating"]))
print(f"{len(dis_view)} sites where the blind configuration disagrees with the reference "
      f"(of {blind_site.ref_true.notna().sum():,} graded sites)")
display(dis_view.round(2))